***

* [总目录](../0_Introduction/0_introduction.ipynb)
* [术语表](../0_Introduction/1_glossary.ipynb)
* [第 2 章：干涉测量的数学工具箱](2_0_introduction.ipynb)
    * 上一节：[2.7 傅里叶定理：移位、缩放与卷积的统一语言](2_7_fourier_theorems.ipynb)
    * 下一节：[2.9 采样理论：离散测量的分辨率与混叠](2_9_sampling_theory.ipynb)

***


导入标准模块:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
try:
    from IPython.display import HTML
except ImportError:
    def HTML(*args, **kwargs):
        return None
%matplotlib inline
HTML('../style/course.css')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12


导入本节所需的专用模块:

In [ ]:
try:
    from IPython.display import HTML
except ImportError:
    def HTML(*args, **kwargs):
        return None
HTML('../style/code_toggle.html')


## 2.8 离散傅里叶变换与 FFT：从解析公式到可计算算法<a id='math:sec:the_discrete_fourier_transform_and_the_fast_fourier_transform'></a>


真实数据总是离散的。相关器输出的是有限积分时间、有限频道宽度和有限采样点上的数值，而不是解析形式的连续函数；因此，真正进入计算机的不是连续傅里叶变换，而是它的离散版本。

本节的目标，是把 DFT 和 FFT 理解为“连续理论进入数值实现”的接口，同时说明周期延拓、循环卷积以及计算复杂度这些在实际成像软件中真正起作用的概念。读完这一节后，读者应当能分清楚解析傅里叶变换、离散傅里叶变换以及快速算法之间各自的角色。


### 2.8.1 离散时间傅里叶变换（DTFT）：定义<a id='math:sec:the_discrete_time_fourier_transform_definition'></a>

我们先引入离散时间傅里叶变换（DTFT）。对于一组定义在整数索引上的复序列 $\{y_n \in \mathbb{C}\}_{n\in\mathbb{Z}}$，其 DTFT 定义为


<a id='math:eq:8_001'></a><!--\label{math:eq:8_001}-->$$
Y_{2\pi}(\omega) = \sum_{n\,=\,-\infty}^{\infty} y_n\,e^{-\imath \omega n} \quad \mbox{其中} \quad n \in \mathbb{Z}.
$$

所得结果是关于角频率变量 $\omega$ 的 $2\pi$ 周期函数。这一点非常关键：DTFT 的频域不是“无限不重复的一条直线”，而是一个周期延拓后的频谱。若改用通常的频率变量 $f$，并写 $\omega = 2\pi f$，则可记为


<a id='math:eq:8_002'></a><!--\label{math:eq:8_002}-->$$
Y_{f_s}(f) = \sum_{n\,=\,-\infty}^{\infty} y_n\,e^{-2\pi\imath f t_n},
$$

这里的 $t_n$ 表示采样时刻，下标 $f_s$ 则提醒我们：这个频谱函数的周期由采样频率决定。后面在第 2.9 节会看到，当我们对一个连续信号做离散采样时，DTFT 与 DFT 会自然出现，因此它们不是“凭空定义的离散对象”，而是连续理论进入采样系统后的直接结果。


从计算角度看，DTFT 仍然不是一个可直接执行的有限算法，因为它涉及无限长序列。它的价值主要在于提供一个过渡层：一方面，它保留了连续傅里叶分析的许多直觉；另一方面，它又已经显式体现出离散采样带来的周期性结构。为了把它真正变成可计算对象，我们还需要进一步截断到有限样本，这就引出了 DFT。


#### 2.8.1.1 周期求和与 DTFT <a id='math:sec:Periodic_summation'></a>


周期求和的思想，是把一个非周期函数复制成周期函数。设
$$
g_\tau(t) = \sum_{n=-\infty}^{\infty} g(t+n\tau) = \sum_{n=-\infty}^{\infty} g(t-n\tau),
$$
那么 $g_\tau(t)$ 就是以 $\tau$ 为周期、由无穷多个副本叠加而成的周期函数。频域中也会出现完全类似的结构。对采样后的信号而言，DTFT 产生的正是这种“频谱副本按固定间隔重复出现”的图景：
$$
Y_{f_s}(f) = \sum_{k = -\infty}^{\infty} Y(f-kf_s).
$$

这条公式值得反复体会：采样不会只给出原始频谱 $Y(f)$，而是会在频域中生成一列以 $f_s$ 为间隔排列的副本。后面讨论混叠时，真正发生重叠的正是这些频谱副本。


#### 2.8.1.2 泊松求和公式 <a id='math:sec:Poisson_summation'></a>

把连续傅里叶变换、离散采样和频谱周期复制连接起来的关键结果，就是泊松求和公式。这里我们不追求一般性证明，而只强调它在信号处理中的含义：它告诉我们，时间域上的离散采样与频域上的周期复制是一回事的两种表述。


设连续函数 $y(t)$ 的傅里叶变换为 $Y(f)$，则泊松求和公式可写为


<a id='math:eq:8_003'></a><!--\label{math:eq:8_003}-->$$
\sum_{n = -\infty}^{\infty} \Delta t ~ y(\Delta t n) e^{-2\pi\imath f \Delta t n} = \sum_{k = -\infty}^{\infty} Y(f - \frac{k}{\Delta t}) = \sum_{k = -\infty}^{\infty} Y(f - kf_s) = Y_{f_s}(f). $$

这条等式的左边，是对连续信号在离散采样点上取值后构造出的傅里叶级数；右边，则是连续频谱 $Y(f)$ 在频域中的周期复制。两边描述的是同一个对象，因此它为采样理论提供了最核心的直觉基础。


不过，到这一步为止，我们仍然没有得到一个真正可在计算机上执行的算法，因为式中依然包含无限多个样本项。为了进入数值实现，我们必须进一步把序列限制为有限长度，这就得到下面的离散傅里叶变换。


### 2.8.2 离散傅里叶变换：定义<a id='math:sec:the_discrete_fourier_transform_definition'></a>


现在令
$$
y = \left\{y_n \in \mathbb{C}\right\}_{n = 0, \ldots, N-1}
$$
是一组长度为 $N$ 的有限复序列。它的离散傅里叶变换（DFT）定义为


<a id='math:eq:8_004'></a><!--\label{math:eq:8_004}-->$$
\mathscr{F}_{\rm D}: \left\{y_n \in \mathbb{C}\right\}_{n \,=\, 0, \ldots, N-1} \rightarrow \left\{Y_k \in \mathbb{C}\right\}_{k \,=\, 0, \ldots, N-1}\\
\mathscr{F}_{\rm D}\{y\} = \left\{Y_k\in\mathbb{C}\right\}_{k \,=\, 0, \ldots, N-1} \quad \mbox{其中} \quad 
Y_k = \sum_{n\,=\,0}^{N-1} y_n\,e^{-2\pi\imath f_k t_n} = \sum_{n\,=\,0}^{N-1} y_n\,e^{-\imath 2\pi \frac{nk}{N}}.
$$

在均匀采样情形下，时间坐标与频率坐标通常写成


$$ t_n = t_0 + n\Delta t \quad \mbox{and} \quad f_k = \frac{kf_s}{N} \quad \mbox{其中} \quad f_s = \frac{1}{\Delta t}. $$

在这种写法中，指数核被整理成 $nk/N$ 的形式，这使许多代数推导更加透明。它还清楚地表明：DFT 输出的并不是任意连续频率上的函数值，而是恰好 $N$ 个离散频率采样点上的复系数。换句话说，DFT 处理的是“有限长度序列”与“有限个频率通道”之间的映射。


$$ \mathscr{F}_{\rm D}\{y\}_k = Y_k =  \sum_{n\,=\,0}^{N-1} y_n\,e^{-\imath 2\pi \frac{nk}{N}}, $$

这里下标 $k$ 表示“取第 $k$ 个频率分量”，而不是参与求和的自由变量。序列 $y_n$ 与 $Y_k$ 互为离散傅里叶对；在后面的成像与谱分析里，我们会不断在这两个表示之间来回切换。


DFT 的逆变换把这 $N$ 个频域系数重新映回长度为 $N$ 的原始序列：


<a id='math:eq:8_005'></a><!--\label{math:eq:8_005}-->$$
\mathscr{F}_{\rm D}^{-1}: \left\{Y_k \in \mathbb{C}\right\}_{k \,=\, 0, \ldots, N-1} \rightarrow \left\{y_n \in \mathbb{C}\right\}_{n \,=\, 0, \ldots, N-1}\\
\mathscr{F}_{\rm D}^{-1}\{Y\} = \left\{y_n\in\mathbb{C}\right\}_{n = 0, \ldots, N-1}
\quad \mbox{其中} \quad y_n = \frac{1}{N} \sum_{k \ = \ 0}^{N-1} Y_k e^{\imath 2\pi \frac{nk}{N}} \ ,
$$

或者简记为


$$ \mathscr{F}_{\rm D}^{-1}\{Y\}_n = y_n = \frac{1}{N} \sum_{k\,=\,0}^{N-1} Y_k\,e^{\imath 2\pi \frac{nk}{N}}. $$

定义中的 $1/N$ 是一种归一化约定；不同教材会把这个因子放到正变换、逆变换，或者分别平分到两边。只要一套材料内部保持一致，这些约定并不会改变物理内容。更值得注意的是：DFT 天然对应一个有限观测窗口，因此它隐含了“序列被截断”和“频谱只在离散频点上取样”这两层数值语境。


<a id='math:eq:8_006'></a><!--\label{math:eq:8_006}-->$$
\begin{align}
\mathscr{F}_{\rm D}^{-1}\left\{\mathscr{F}_{\rm D}\left\{y\right\}\right\}_{n^\prime} \,&=\, \frac{1}{N}\sum_{k\,=\,0}^{N-1} \left(\sum_{n\,=\,0}^{N-1} y_n e^{-\imath 2\pi\frac{kn}{N}}\right)e^{\imath 2\pi\frac{kn^\prime}{N}}\\
&=\,\frac{1}{N}\sum_{k\,=\,0}^{N-1} \sum_{n\,=\,0}^{N-1} \left( y_n e^{-\imath 2\pi\frac{kn}{N}}e^{\imath 2\pi\frac{kn^\prime}{N}}\right)\\
&=\,\frac{1}{N}\left(\sum_{k\,=\,0}^{N-1} y_{n^\prime}+\sum_{\begin{split}n\,&=\,0\\n\,&\neq\,n^\prime\end{split}}^{N-1} \sum_{k\,=\,0}^{N-1} y_n e^{-\imath 2\pi\frac{kn}{N}}e^{\imath 2\pi\frac{kn^\prime}{N}}\right)\\
&=\,\frac{1}{N}\left(\sum_{k\,=\,0}^{N-1} y_{n^\prime}+\sum_{\begin{split}n\,&=\,0\\n\,&\neq\,n^\prime\end{split}}^{N-1} \sum_{k\,=\,0}^{N-1} y_n e^{\imath 2\pi\frac{k(n^\prime-n)}{N}}\right)\\
&=\,y_{n^\prime}+\frac{1}{N}\sum_{\begin{split}n\,&=\,0\\n\,&\neq\,n^\prime\end{split}}^{N-1} y_n \sum_{k\,=\,0}^{N-1} \left(e^{\imath 2\pi\frac{(n^\prime-n)}{N}}\right)^k\\
&=\,y_{n^\prime}+\frac{1}{N}\sum_{\begin{split}n\,&=\,0\\n\,&\neq\,n^\prime\end{split}}^{N-1} y_n \frac{1-\left(e^{\imath 2\pi\frac{(n^\prime-n)}{N}}\right)^N}{1-\left(e^{\imath 2\pi\frac{(n^\prime-n)}{N}}\right)}\\
&=\,y_{n^\prime}+\frac{1}{N}\sum_{\begin{split}n\,&=\,0\\n\,&\neq\,n^\prime\end{split}}^{N-1} y_n \frac{1-e^{\imath 2\pi(n^\prime-n)}}{1-e^{\imath 2\pi\frac{(n^\prime-n)}{N}}}\\
&\underset{n,n^\prime \in \mathbb{N}}{=}\,y_{n^\prime},\\
\end{align}
$$

上面的推导用到了有限几何级数求和以及复指数基底的离散正交性。它们共同保证：在这组离散基底上展开、再按逆变换合成，能够精确恢复原始有限序列。


与连续傅里叶变换类似，DFT 也具有周期性，但这种周期性现在直接体现在索引空间里：频域索引 $k$ 和原始域索引 $n$ 都只在模 $N$ 的意义下有区别。


<a id='math:eq:8_007'></a><!--\label{math:eq:8_007}-->$$
\begin{align}
\mathscr{F}_{\rm D}\{y \}_k \,&=\,\mathscr{F}_{\rm D}\{y \}_{k \pm N}  \\
\mathscr{F}_{\rm D}^{-1}\{Y \}_{n} \,&=\,\mathscr{F}_{\rm D}^{-1}\{Y \}_{n \pm N}.\\
\end{align}
$$

与连续情形类似，逆 DFT 也可以改写成一个带索引反号的正向 DFT：


<a id='math:eq:8_008'></a><!--\label{math:eq:8_008}-->$$
\begin{align}
\mathscr{F}_{\rm D}^{-1}\{Y\}_n \,&=\, \frac{1}{N} \mathscr{F}_{\rm D}\{Y\}_{-n} \\
&=\,\frac{1}{N} \mathscr{F}_{\rm D}\{Y\}_{N-n}.\\
\end{align}
$$

若输入序列 $y_n$ 为实数，则其 DFT 具有厄米共轭对称性；反过来，满足该对称性的复频谱也对应实值时域序列。这一性质在只存储“正频率一半谱线”时非常有用：


<a id='math:eq:8_009'></a><!--\label{math:eq:8_009}-->$$
\begin{split}
\mathscr{F}_{\rm D}\{y\}_k\,&=\, \left(\mathscr{F}_{\rm D}\{y\}_{-k}\right)^*\\
&=\, \left(\mathscr{F}_{\rm D}\{y\}_{N-k}\right)^* \ .
\end{split}
$$

### 2.8.3 离散卷积：定义与离散卷积定理<a id='math:sec:the_discrete_convolution_definition_and_discrete_convolution_theorem'></a>


与解析卷积类似，对于两系列数$y = \left\{y_n \in \mathbb{C}\right\}_{n = 0, \ldots, N-1}$ 和 $z = \left\{z_n \in \mathbb{C}\right\}_{n = 0, \ldots, N-1}$，离散卷积定义如下

<a id='math:eq:8_010'></a><!--\label{math:eq:8_010}-->$$
\circ: \left\{y_n \in \mathbb{C}\right\}_{n \,=\, 0, \ldots, N-1}\times \left\{z_n \in \mathbb{C}\right\}_{n \,=\, 0, \ldots, N-1} \rightarrow \left\{r_k \in \mathbb{C}\right\}_{k \,=\,  0, \ldots, N-1}\\
(y\circ z)_k = r_k = \sum_{n\,=\,0}^{N-1} y_n z_{k-n}.\\
$$

然而，这个定义有点微妙。我们必须考虑,如果$n > k$;索引$k-n$将是负的。由于我们将指数定义为严格正的，这就需要引入有时被称为“概括”的约定。回想复数$r_k = e^{\frac{\imath 2\pi k}{N}}$有这一性质：$r_{k \pm mN} = r_k$, 其中 $m \in \mathbb{Z}$是个整数

$$ (y\circ z)_k = r_k = \sum_{n\,=\,0}^{N-1} y_n z_{(k-n) \, \text{mod} \, N}, $$

其中mod表示模运算。就像普通卷积一样，离散卷积是可交换的。从这个方程中可以明显看出一个重要的效果，如果两个级数足够“宽”，卷积将在级数开始时继续进行，这一效果称为混叠。

卷积定理在 DFT 框架下同样成立。也就是说，一个域中的循环卷积对应另一个域中的逐点乘积；反过来，一个域中的逐点乘积则对应另一个域中的循环卷积。设 $(y \odot z)_n \underset{def}{=} y_n z_n$ 为 Hadamard（元素逐位）乘积，那么对傅里叶对 $Y_k \leftrightarrow y_n$ 与 $Z_k \leftrightarrow z_n$，有


<a id='math:eq:8_011'></a><!--\label{math:eq:8_011}-->$$
\forall N\,\in\, \mathbb{N}\\
\begin{align}
y \,&=\, \left\{y_n \in \mathbb{C}\right\}_{n\,=\,0, \ldots, \,N-1}\\
z \,&=\, \left\{z_n \in \mathbb{C}\right\}_{n\,=\,0, \ldots, \,N-1}\\
Y \,&=\, \left\{Y_k \in \mathbb{C}\right\}_{k\,=\,0, \ldots, \,N-1}\\
Z \,&=\, \left\{Z_k \in \mathbb{C}\right\}_{k\,=\,0, \ldots, \,N-1}\\
\end{align}\\
\begin{split}
\mathscr{F}_{\rm D}\{y\odot z\}\,&=\,\frac{1}{N}\mathscr{F}_{\rm D}\{y\}\circ \mathscr{F}_{\rm D}\{z\}\\
\mathscr{F}_{\rm D}^{-1}\{Y\odot Z\}\,&=\,\mathscr{F}_{\rm D}\{Y\}\circ \mathscr{F}_{\rm D}\{Z\}\\
\mathscr{F}_{\rm D}\{y\circ z\}\,&=\,\mathscr{F}_{\rm D}\{y\} \odot \mathscr{F}_{\rm D}\{z\}\\
\mathscr{F}_{\rm D}^{-1}\{Y\circ Z\}\,&=\,\frac{1}{N}\mathscr{F}_{\rm D}\{Y\} \odot \mathscr{F}_{\rm D}\{Z\}\\
\end{split}
$$

#### 代码演示：循环卷积、边界回绕与零填充

上面的离散卷积定理最容易在实践中被误解。对长度固定为 $N$ 的 DFT 而言，频域逐点相乘对应的并不是普通意义下的“无限线性卷积”，而是**模 $N$ 的循环卷积**。这意味着如果卷积结果在右端超出数组边界，它会从左端“绕回”到序列开头。

这件事在后续图像处理和 FFT 卷积里非常关键：若想让 FFT 给出与普通线性卷积一致的结果，通常必须先做足够的零填充（zero padding）。否则，边界回绕会把本不该相遇的结构混合在一起。


In [ ]:
kernel = np.array([0.05, 0.20, 0.50, 0.20, 0.05])
N = 32
n = np.arange(N)

x = np.zeros(N)
x[7] = 0.6
x[28] = 1.0

h = np.zeros(N)
h[:kernel.size] = kernel

circular_conv = np.real(np.fft.ifft(np.fft.fft(x) * np.fft.fft(h)))
nfft = N + kernel.size - 1
linear_conv_fft = np.real(np.fft.ifft(np.fft.fft(x, nfft) * np.fft.fft(kernel, nfft)))
linear_conv_direct = np.convolve(x, kernel, mode='full')

print(f'Max difference between zero-padded FFT and direct linear convolution = {np.max(np.abs(linear_conv_fft - linear_conv_direct)):.2e}')

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

axes[0].stem(n, x, linefmt='C0-', markerfmt='C0o', basefmt=' ')
axes[0].stem(np.arange(kernel.size), kernel, linefmt='C1-', markerfmt='C1s', basefmt=' ')
axes[0].set_xlabel('Sample index')
axes[0].set_ylabel('Amplitude')
axes[0].set_title('Input sequence and compact kernel')
axes[0].grid(alpha=0.3)

axes[1].stem(n, circular_conv, linefmt='C2-', markerfmt='C2o', basefmt=' ')
axes[1].axvspan(0, kernel.size - 1, color='tab:red', alpha=0.12)
axes[1].set_xlim(0, N - 1)
axes[1].set_xlabel('Sample index')
axes[1].set_ylabel('Amplitude')
axes[1].set_title('Circular convolution from an N-point DFT')
axes[1].grid(alpha=0.3)

axes[2].stem(np.arange(nfft), linear_conv_direct, linefmt='C0-', markerfmt='C0o', basefmt=' ', label='direct linear convolution')
axes[2].plot(np.arange(nfft), linear_conv_fft, 'C3--', lw=2, label='zero-padded FFT result')
axes[2].set_xlabel('Sample index')
axes[2].set_ylabel('Amplitude')
axes[2].set_title('Zero padding recovers the linear convolution')
axes[2].grid(alpha=0.3)
axes[2].legend(fontsize=9)

plt.tight_layout()
plt.show()


这个例子把三件事一次看清了。第一，DFT 默认把长度为 $N$ 的序列看成一个周期信号，因此卷积天然带有回绕；第二，频域逐点相乘并不会自动给出“普通线性卷积”；第三，只要零填充长度至少达到 $N+M-1$，FFT 卷积就能与直接线性卷积一致。

在干涉成像里，这种差别会反复出现。例如，有限网格上的 FFT、卷积核支撑范围和图像边界处理，都会决定边界能否被安全地隔离开。把“循环卷积”和“线性卷积”的区别搞清楚，是理解后续 gridding、PSF 计算与快速卷积实现的前提。


### 2.8.4 DFT 的数值实现 <a id='math:sec:numerical_DFT'></a>


现在我们来看看DFT是如何在数字上实现的。最直接的方法是以双循环形式对组件求和

In [ ]:
def loop_DFT(x):
    """
    Implementing the DFT in a double loop
    Input: x = the vector we want to find the DFT of
    """
    #Get the length of the vector (will only work for 1D arrays)
    N = x.size
    #Create vector to store result in
    X = np.zeros(N,dtype=complex)
    for k in range(N):
        for n in range(N):
            X[k] += np.exp(-1j*2.0*np.pi*k*n/N)*x[n]
    return X

虽然这将产生正确的结果，但这种实现DFT的方法将非常慢。DFT可以用矩阵形式实现。要确信，这个操作可以用向量化实现

其中$K$是矩阵核，它存储的值为 $K_{kn} = e^{\frac{-\imath 2 \pi k n}{N}}$。它的数值实现如下

In [ ]:
def matrix_DFT(x):
    """
    Implementing the DFT in vectorised form
    Input: x = the vector we want to find the DFT of
    """
    #Get the length of the vector (will only work for 1D arrays)
    N = x.size
    #Create vector to store result in
    n = np.arange(N)
    k = n.reshape((N,1))
    K = np.exp(-1j*2.0*np.pi*k*n/N)
    return K.dot(x)

这个函数将比以前的实现快得多。我们应该检查它们是否返回相同的结果

In [ ]:
x = np.random.random(256)  #create random vector to take the DFT of
np.allclose(loop_DFT(x),matrix_DFT(x)) #compare the result using numpy's built in function

了确保DFT确实有效，我们还将函数的输出与内置在DFT函数中的numpy的输出进行比较(注意，numpy自动实现了一个名为FFT的DFT的更快版本，参见下面的讨论)

In [ ]:
x = np.random.random(256)  #create random vector to take the DFT of
np.allclose(np.fft.fft(x),matrix_DFT(x)) #compare the result using numpy's built in function

太棒了!我们的函数返回正确的结果。接下来，我们做一个例子来演示函数的频谱(频域)表示和时间(时域)表示之间的对偶性。如下例所示，时间序列的傅里叶变换返回信号中包含的频率。

下面的代码模拟信号

采用DFT和画出信号的振幅和相位

In [ ]:
#First we simulate a time series as the sum of a number of sinusoids each with a different frequency
N = 512  #The number of samples of the time series
tmin = -10 #The minimum value of the time coordinate
tmax = 10 #The maximum value of the time coordinate
t = np.linspace(tmin,tmax,N) #The time coordinate
f1 = 1.0 #The frequency of the first sinusoid
f2 = 2.0 #The frequency of the second sinusoid
f3 = 3.0 #The frequency of the third sinusoid
#Generate the signal
y = np.sin(2.0*np.pi*f1*t) + np.sin(2.0*np.pi*f2*t) + np.sin(2.0*np.pi*f3*t)
#Take the DFT
Y = matrix_DFT(y)
#Plot the absolute value, real and imaginary parts
plt.figure(figsize=(15, 6))
plt.subplot(121)
plt.stem(abs(Y))
plt.xlabel('$k$',fontsize=18)
plt.ylabel(r'$|Y_k|$',fontsize=18)
plt.subplot(122)
plt.stem(np.angle(Y))
plt.xlabel('$k$',fontsize=18)
plt.ylabel(r'phase$(Y_k)$',fontsize=18)

**图 2.8.1：** 由三个不同频率正弦分量叠加而成的信号，其 DFT 幅度与相位示意。


若只把横轴看成频率索引 $k$，这三个分量并不总是立刻显而易见。更自然的做法，是把频率索引换算成真正的物理频率
$$
f_k = \frac{k f_s}{N},
$$
其中 $f_s$ 是采样频率。下面我们就把谱线画在 $f_k$ 上，而不是仅画在整数索引上。


In [ ]:
#Get the sampling frequency
delt = t[1] - t[0]
fs = 1.0/delt
k = np.arange(N)
fk = k*fs/N
plt.figure(figsize=(15, 6))
plt.subplot(121)
plt.stem(fk,abs(Y))
plt.xlabel('$f_k$',fontsize=18)
plt.ylabel(r'$|Y_k|$',fontsize=18)
plt.subplot(122)
plt.stem(fk,np.angle(Y))
plt.xlabel('$f_k$',fontsize=18)
plt.ylabel(r'phase$(Y_k)$',fontsize=18)

**图 2.8.2：** 使用物理频率 $f_k$ 作为横轴标记的离散频谱。


现在可以清楚看到，三个主峰对应于输入信号中的三个频率分量 $f_1 = 1\,\mathrm{Hz}$、$f_2 = 2\,\mathrm{Hz}$ 与 $f_3 = 3\,\mathrm{Hz}$。至于其余峰值，则需要结合 DFT 的两个基本性质来理解：

* 对实值信号，频谱满足厄米共轭对称，因此负频率部分并没有提供新的独立信息；
* DFT 在索引空间里以 $N$ 为周期，因此频谱会在离散频率轴上重复出现。

两者合起来就得到常见关系 $Y_{N-k}=Y_k^*$。这也是为什么对实值输入而言，通常只需要保留前 $N/2+1$ 个频率分量。


不过，这个例子里还有两个现象尚未解释：

* 某些并非输入真频率的位置上，为什么也会出现非零谱值？
* 为什么三个主峰的高度不一定完全相同？

答案都与有限采样有关。只要观测窗口有限、采样点数有限，频谱泄漏、有限分辨率和离散频点错位就不可避免。第 2.9 节会把这些现象系统化为采样定理、混叠和窗口效应；在这里，读者可以先通过调节 $N$、$t_{\min}$、$t_{\max}$ 以及输入频率，建立对这些数值效应的直觉。


我们已经看到，矢量化实现的 DFT 会比显式双循环快得多。下面用简单的计时命令直观看看两者相差多少。


In [ ]:
%timeit loop_DFT(x)
%timeit matrix_DFT(x)

通常可以看到接近一个数量级的差异。接下来再把它与 `numpy` 基于 FFT 的实现进行比较。


In [ ]:
%timeit np.fft.fft(x)

结果往往会非常显著：`numpy` 的 `FFT` 实现通常比我们手写的向量化 DFT 再快几个数量级。问题自然变成了：这种加速是如何做到的？


### 2.8.5 快速傅里叶变换<a id='math:sec:fast_fourier_tranforms'></a>


DFT 的直接实现计算代价很高。无论是双循环版本，还是仅仅把矩阵乘法交给底层库的向量化版本，本质上都仍然需要处理 $N^2$ 量级的复指数与乘法，因此复杂度是 $\mathcal{O}(N^2)$。

FFT 的关键思想，是利用 DFT 核函数中的对称性与重复结构，把一个长度为 $N$ 的 DFT 递归拆分为若干个更短的 DFT。最经典的 Cooley-Tukey 思路，就是把序列分成偶数索引部分与奇数索引部分，再把原问题化为两个长度约为 $N/2$ 的子问题。这样不断递归下去，最终就能把总复杂度降到 $\mathcal{O}(N\log N)$。

这并不改变 DFT 的数学定义；FFT 只是更高效地计算同一个结果。因此，理解 FFT 的最好方式，不是把它看成“另一种变换”，而是把它看成 DFT 的快速算法实现。


In [ ]:
def one_layer_FFT(x):
    """An implementation of the 1D Cooley-Tukey FFT using one layer."""
    N = x.size
    if N % 2 > 0:
        print('Warning: length of x is odd, returning the DFT instead.')
        return matrix_DFT(x)
    X_even = matrix_DFT(x[::2])
    X_odd = matrix_DFT(x[1::2])
    factor = np.exp(-2j * np.pi * np.arange(N) / N)
    return np.concatenate([
        X_even + factor[:N // 2] * X_odd,
        X_even + factor[N // 2:] * X_odd,
    ])


下面通过与 `numpy` 的 `FFT` 结果比较，确认这个单层 Cooley-Tukey 实现确实返回了与 DFT 一致的谱值。


In [ ]:
np.allclose(np.fft.fft(x),one_layer_FFT(x))

这个简单示例已经揭示了 FFT 的核心收益：通过重用 DFT 中的大量重复结构，我们不再需要按朴素方式执行全部 $N^2$ 次运算。真正工业级的 FFT 实现还会进一步利用缓存布局、实序列对称性、混合基数分解等技巧，因此性能会比课堂版示例更高。

对本书而言，最重要的不是手写最快的 FFT，而是理解它对后续数据处理意味着什么：许多看似不可能的大规模傅里叶计算之所以能在成像、卷积和谱分析里成为日常操作，依赖的正是 FFT 这一层算法工程。


***

* 下一节：[2.9 采样理论：离散测量的分辨率与混叠](2_9_sampling_theory.ipynb)
